# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset with mlcroissant
dataset = mlc.Dataset(croissant_url)

# Print out the dataset metadata: name and description
print(f"Dataset title: {dataset.metadata.name}\nDescription: {dataset.metadata.description}\n")
print(f"Published: {dataset.metadata.datePublished}  |  License: {dataset.metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Croissant datasets are organized in *record sets* (tables/files), each with unique `@id`s.
Below we enumerate the record sets, their fields, and columns.

In [ ]:
# List all record sets in the dataset, their @ids and included fields
record_sets = dataset.metadata.recordSets # This is a list of RecordSet objects.
if not record_sets:
    print("No record sets found in the Croissant schema.")
else:
    for rset in record_sets:
        print(f"Record set name: {rset.name}")
        print(f"  @id: {rset.id}")
        print("  Fields and columns:")
        for field in rset.fields:
            print(f"    Field name: {getattr(field, 'name', None)}  |  @id: {getattr(field, 'id', None)}")
            if hasattr(field, 'columns') and field.columns:
                for col in field.columns:
                    print(f"      Column name: {getattr(col, 'name', None)}  |  @id: {getattr(col, 'id', None)}")
        print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.
Below, we demonstrate loading all record sets, using their `@id`s, and loading a sample of records for inspection.

In [ ]:
# Collect all record set @ids
record_sets = dataset.metadata.recordSets
record_set_ids = [rset.id for rset in record_sets]

# Load each record set as a DataFrame
dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records from record set {record_set_id}")
        print(f"Columns: {df.columns.tolist()}")
        print(df.head())
    except Exception as e:
        print(f"Could not load records for {record_set_id}: {str(e)}")

# For demonstration, select the first record set for further analysis
if record_set_ids:
    first_record_set_id = record_set_ids[0]
    print(f"Will use record set: {first_record_set_id}")
    df = dataframes[first_record_set_id]
else:
    first_record_set_id = None
    df = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering records based on specific criteria, normalizing numeric fields, grouping data for summary statistics, etc.

**Note**: All fields and columns are referenced by their Croissant schema `@id`.

In [ ]:
import numpy as np

if df is not None and not df.empty:
    print("Columns in this DataFrame:\n", df.columns.tolist())
    # Attempt to select a numeric field for demonstration
    # Pick the first numerical column (float64/int64)
    numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_columns:
        print("No numeric columns found to analyze.")
    else:
        numeric_field = numeric_columns[0] # Use the first found
        print(f"Selected numeric field for filtering: {numeric_field}")

        threshold = df[numeric_field].mean() # Use the mean as threshold
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize numeric field
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to group by any object-type field
        object_columns = df.select_dtypes(include=["object"]).columns.tolist()
        group_field = None
        for possible in object_columns:
            if df[possible].nunique() < len(df) and df[possible].nunique() > 1:
                group_field = possible
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by {group_field}:")
            print(grouped_df.head())
else:
    print("No data loaded to analyze. Please check that the record set fields are properly loaded.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

- Histograms of numeric fields
- Scatter plot if at least two numeric fields are present

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and not df.empty:
    numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()
    if len(numeric_columns) >= 1:
        # Plot histogram of the first numeric column
        plt.figure(figsize=(6,4))
        sns.histplot(df[numeric_columns[0]], bins=30, kde=True)
        plt.title(f"Distribution of {numeric_columns[0]}")
        plt.xlabel(numeric_columns[0])
        plt.show()
    if len(numeric_columns) >= 2:
        # Scatter plot between first two numeric columns
        plt.figure(figsize=(6,4))
        sns.scatterplot(x=df[numeric_columns[0]], y=df[numeric_columns[1]])
        plt.title(f"Scatter plot of {numeric_columns[0]} vs. {numeric_columns[1]}")
        plt.xlabel(numeric_columns[0])
        plt.ylabel(numeric_columns[1])
        plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion
In this notebook, we loaded and explored the [Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset using the `mlcroissant` library, starting from schema navigation to analysis and visualization.

- All data entities (record sets, fields, columns) were referenced by their `@id` as required by the Croissant specification.
- We demonstrated how to extract data loaded from each record set, explore fields, filter and transform numeric attributes, and visualize the distribution or relationships present in the data.
- For deeper analysis, consult the [FAIR² dataset's documentation](https://sen.science/doi/10.71728/senscience.vq4a-28xa) and the record/field structure listed above.